<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/libros/ciencia-de-datos-con-python/volumen-01/cuadernos/capitulo-009-valores-extremos-y-primeros-rankings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Capítulo 9 — Valores extremos y primeros rankings

## Breve repaso

En el trabajo anterior sobre ordenamiento aprendimos a reorganizar las filas de un `DataFrame` usando `sort_values()`.

Vimos que ordenar datos permite leer mejor una tabla, identificar valores altos o bajos, agrupar visualmente registros relacionados y presentar la información de una manera más clara. También aprendimos a ordenar de forma ascendente o descendente, a ordenar por más de una columna y a combinar filtros con ordenamiento.

En este capítulo vamos a usar esas ideas para resolver un tipo de pregunta muy frecuente en el análisis de datos: la búsqueda de valores extremos y primeros rankings.

Muchas veces no necesitamos ver todo el dataset ordenado. Tal vez solo queremos conocer los productos más caros, los más baratos, las ventas con mayor cantidad de unidades o los registros que aparecen al final de cierto ordenamiento.

Para eso vamos a trabajar con herramientas como `head()`, `tail()`, `nlargest()` y `nsmallest()`.

Estas herramientas nos permiten construir respuestas más directas a preguntas como:

```text
¿Cuáles son los productos más caros?
¿Cuáles son los productos más baratos?
¿Qué ventas tuvieron mayor cantidad vendida?
¿Qué ventas tuvieron menor cantidad vendida?
¿Cuáles son los primeros registros después de ordenar?
¿Cuáles son los últimos registros después de ordenar?
```

Al finalizar este capítulo deberías poder:

* Usar `head()` para ver los primeros registros de un `DataFrame`.
* Usar `tail()` para ver los últimos registros de un `DataFrame`.
* Combinar `sort_values()` con `head()` y `tail()`.
* Usar `nlargest()` para obtener los mayores valores de una columna.
* Usar `nsmallest()` para obtener los menores valores de una columna.
* Construir rankings simples a partir de distintas variables.
* Combinar filtros, ordenamiento y selección de columnas para obtener resultados más claros.
* Reconocer cuándo conviene usar `sort_values()` y cuándo conviene usar `nlargest()` o `nsmallest()`.



## Dataset de trabajo

Para estudiar valores extremos y primeros rankings vamos a seguir trabajando con el dataset de ventas de una tienda escolar. Cada fila representa una venta registrada. Las columnas indican el producto vendido, la categoría, el precio unitario, la cantidad vendida, la sucursal, la forma de pago y la fecha.

Este dataset nos permite buscar distintos tipos de extremos: productos más caros, productos más baratos, ventas con mayor cantidad vendida y ventas con menor cantidad vendida.

In [ ]:
import pandas as pd

datos = {
    "producto": [
        "Cuaderno", "Lapicera", "Mochila", "Regla", "Cartuchera",
        "Calculadora", "Auriculares", "Resaltadores", "Guardapolvo", "Compas",
        "Pendrive", "Carpeta", "Goma", "Lapiz", "Agenda"
    ],
    "categoria": [
        "Libreria", "Libreria", "Accesorios", "Libreria", "Accesorios",
        "Tecnologia", "Tecnologia", "Libreria", "Indumentaria", "Libreria",
        "Tecnologia", "Libreria", "Libreria", "Libreria", "Libreria"
    ],
    "precio": [
        1200, 350, 8500, 500, 3200,
        18500, 7600, 2100, 14500, 2600,
        9800, 950, 300, 250, 4300
    ],
    "cantidad_vendida": [
        15, 40, 5, 25, 8,
        3, 6, 12, 4, 10,
        7, 18, 30, 50, 9
    ],
    "sucursal": [
        "Centro", "Centro", "Norte", "Sur", "Norte",
        "Centro", "Sur", "Centro", "Norte", "Sur",
        "Centro", "Norte", "Sur", "Centro", "Sur"
    ],
    "forma_pago": [
        "Efectivo", "Debito", "Credito", "Efectivo", "Debito",
        "Credito", "Credito", "Debito", "Credito", "Efectivo",
        "Debito", "Efectivo", "Efectivo", "Debito", "Credito"
    ],
    "fecha": [
        "2024-03-01", "2024-03-01", "2024-03-02", "2024-03-02", "2024-03-03",
        "2024-03-03", "2024-03-04", "2024-03-04", "2024-03-05", "2024-03-05",
        "2024-03-06", "2024-03-06", "2024-03-07", "2024-03-07", "2024-03-08"
    ]
}

df = pd.DataFrame(datos)

df

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
9,Compas,Libreria,2600,10,Sur,Efectivo,2024-03-05


Al observar el `DataFrame`, vemos que hay productos con precios muy diferentes y cantidades vendidas también variadas. Esa diversidad hace que tenga sentido buscar valores extremos.

Por ejemplo, podemos preguntarnos qué productos tienen los precios más altos, cuáles tienen los precios más bajos, qué registros muestran mayor cantidad vendida o cuáles aparecen al final de un determinado ordenamiento.

## Qué significa buscar valores extremos

Cuando analizamos datos, muchas veces nos interesan especialmente los valores más altos o más bajos de una variable.

Por ejemplo, en nuestro dataset podríamos buscar el producto con mayor precio, el producto con menor precio, la venta con mayor cantidad de unidades o la venta con menor cantidad de unidades. Estos casos se conocen como valores extremos porque se ubican en los límites de una variable: los más grandes o los más pequeños.

Buscar valores extremos no significa necesariamente que esos valores sean errores. Un precio muy alto puede ser perfectamente válido si corresponde a un producto caro, como una calculadora o un guardapolvo. Una cantidad vendida muy baja también puede ser razonable si se trata de un producto de mayor precio o de menor demanda.

Lo importante es que los valores extremos llaman nuestra atención y nos invitan a observar con más cuidado. A veces nos ayudan a construir rankings, a detectar productos destacados o a identificar registros que conviene revisar.

En Pandas podemos encontrar estos valores de varias maneras. Una estrategia es ordenar el `DataFrame` y mirar las primeras o últimas filas. Otra posibilidad es usar métodos específicos como `nlargest()` y `nsmallest()`, que permiten obtener directamente los mayores o menores valores de una columna.

## Primeros registros con head()

El método `head()` permite ver las primeras filas de un `DataFrame`.

Ya lo usamos antes para inspeccionar rápidamente una tabla. En este capítulo lo vamos a retomar con otro propósito: observar los primeros registros de un resultado ordenado.

Primero recordemos su uso básico:

In [ ]:
df.head()

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03


Por defecto, `head()` muestra las primeras cinco filas.

También podemos indicar cuántas filas queremos ver:

In [ ]:
df.head(3)

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02


En este caso, Pandas muestra las primeras tres filas del `DataFrame`.

Por sí solo, `head()` no busca los valores más altos ni los más bajos. Simplemente muestra las primeras filas según el orden actual de la tabla.

Por eso, si queremos usar `head()` para construir un ranking, primero necesitamos ordenar los datos.

## Últimos registros con tail()

El método `tail()` permite ver las últimas filas de un `DataFrame`.

Su uso es similar al de `head()`, pero en lugar de mostrar el comienzo de la tabla, muestra el final.

In [ ]:
df.tail()

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06
11,Carpeta,Libreria,950,18,Norte,Efectivo,2024-03-06
12,Goma,Libreria,300,30,Sur,Efectivo,2024-03-07
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07
14,Agenda,Libreria,4300,9,Sur,Credito,2024-03-08


Por defecto, `tail()` muestra las últimas cinco filas.

También podemos indicar una cantidad específica:

In [ ]:
df.tail(3)

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
12,Goma,Libreria,300,30,Sur,Efectivo,2024-03-07
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07
14,Agenda,Libreria,4300,9,Sur,Credito,2024-03-08


Al igual que `head()`, `tail()` no calcula por sí mismo valores extremos. Solo muestra las últimas filas según el orden actual del `DataFrame`.

Esto significa que `head()` y `tail()` se vuelven especialmente útiles cuando los combinamos con un ordenamiento previo.

## Ordenar y luego usar head()

Si queremos obtener los valores más altos de una columna, podemos ordenar el `DataFrame` de forma descendente y luego usar `head()`.

Por ejemplo, para ver los cinco productos con mayor precio, podemos escribir:

In [ ]:
df.sort_values("precio", ascending=False).head()

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04


Esta instrucción combina dos pasos.

Primero, `sort_values("precio", ascending=False)` ordena el `DataFrame` por precio de mayor a menor. Luego, `head()` muestra las primeras cinco filas de ese resultado ordenado.

De esta manera obtenemos un primer ranking de productos según su precio.

Si queremos ver solo los tres productos más caros, podemos indicar ese número dentro de `head()`.

In [ ]:
df.sort_values("precio", ascending=False).head(3)

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06


Ahora el resultado muestra solamente las tres ventas con precios más altos.

También podemos aplicar la misma idea con la columna `cantidad_vendida`. Si queremos ver las ventas con mayor cantidad de unidades, ordenamos por `cantidad_vendida` en forma descendente y luego usamos `head()`.

In [ ]:
df.sort_values("cantidad_vendida", ascending=False).head()

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
12,Goma,Libreria,300,30,Sur,Efectivo,2024-03-07
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02
11,Carpeta,Libreria,950,18,Norte,Efectivo,2024-03-06


Esta forma de trabajo es muy frecuente: primero ordenamos según la variable que nos interesa y después mostramos los primeros registros del resultado.

## Ordenar y luego usar tail()

También podemos usar `tail()` después de ordenar.

Si ordenamos una columna de menor a mayor, las últimas filas serán las que tienen los valores más altos.

Por ejemplo:

In [ ]:
df.sort_values("precio").tail()

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03


En este caso, `sort_values("precio")` ordena los precios de menor a mayor. Luego, `tail()` muestra las últimas cinco filas, que corresponden a los precios más altos.

Esto produce un resultado parecido a ordenar de mayor a menor y usar `head()`:

```python
df.sort_values("precio", ascending=False).head()
```

Ambas estrategias pueden servir para encontrar valores altos, aunque suele ser más directo ordenar en forma descendente y usar `head()`.

También podemos usar `tail()` para encontrar valores bajos si ordenamos de mayor a menor:

In [ ]:
df.sort_values("precio", ascending=False).tail()

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
11,Carpeta,Libreria,950,18,Norte,Efectivo,2024-03-06
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
12,Goma,Libreria,300,30,Sur,Efectivo,2024-03-07
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07


En este caso, primero ordenamos los precios de mayor a menor. Luego, `tail()` muestra las últimas filas, que corresponden a los precios más bajos.

Esta combinación es válida, pero debemos leerla con cuidado. Cuando usamos `head()` o `tail()` después de ordenar, siempre conviene preguntarnos:

```text
¿En qué sentido ordené los datos?
¿Estoy mirando el comienzo o el final del resultado?
```

Esa interpretación es necesaria para saber si estamos observando los valores más altos o los más bajos.

## Obtener los mayores valores con nlargest()

Pandas ofrece un método específico para obtener las filas con los mayores valores de una columna: `nlargest()`.

Este método permite pedir directamente los registros más grandes según una variable numérica.

Por ejemplo, si queremos obtener las tres ventas con mayor precio, podemos escribir:

In [ ]:
df.nlargest(3, "precio")

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06


La instrucción anterior devuelve las tres filas con los valores más altos en la columna `precio`.

Esta forma es más directa que ordenar todo el `DataFrame` y luego usar `head()`.

Podemos comparar estas dos instrucciones:

```python
df.sort_values("precio", ascending=False).head(3)
```

y

```python
df.nlargest(3, "precio")
```

En este caso, ambas permiten obtener las tres ventas con mayor precio.

También podemos usar `nlargest()` con otra columna numérica. Por ejemplo, para obtener las cinco ventas con mayor cantidad vendida:

In [ ]:
df.nlargest(5, "cantidad_vendida")

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
12,Goma,Libreria,300,30,Sur,Efectivo,2024-03-07
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02
11,Carpeta,Libreria,950,18,Norte,Efectivo,2024-03-06


`nlargest()` es útil cuando queremos obtener rápidamente los registros con valores más altos en una columna numérica.

## Obtener los menores valores con nsmallest()

Así como `nlargest()` permite obtener los valores más altos, Pandas también ofrece `nsmallest()` para obtener los valores más bajos de una columna numérica.

Por ejemplo, si queremos ver las tres ventas con menor precio, podemos escribir:

In [ ]:
df.nsmallest(3, "precio")

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07
12,Goma,Libreria,300,30,Sur,Efectivo,2024-03-07
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01


La instrucción anterior devuelve las tres filas con los valores más bajos en la columna `precio`.

Esto nos permite responder rápidamente una pregunta como:

```text
¿Cuáles son los productos más baratos del dataset?
```

También podemos usar `nsmallest()` con la columna `cantidad_vendida`:

In [ ]:
df.nsmallest(5, "cantidad_vendida")

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06


En este caso obtenemos las cinco ventas con menor cantidad de unidades vendidas.

Al igual que `nlargest()`, `nsmallest()` trabaja con columnas numéricas. Si queremos ordenar columnas de texto, debemos usar `sort_values()`.

Podemos resumir la diferencia así:

```text
nlargest()   → obtiene los registros con valores más altos

nsmallest()  → obtiene los registros con valores más bajos
```

Estos métodos son muy útiles cuando queremos encontrar extremos sin ordenar visualmente todo el `DataFrame`.

## Comparar formas de buscar extremos

Para buscar valores extremos en Pandas tenemos varias alternativas.

Una posibilidad es ordenar el `DataFrame` y luego mirar las primeras o últimas filas. Por ejemplo:

In [ ]:
df.sort_values("precio", ascending=False).head(3)

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06


Esta instrucción ordena todo el `DataFrame` por `precio`, de mayor a menor, y luego muestra las primeras tres filas.

Otra posibilidad es usar directamente `nlargest()`:

In [ ]:
df.nlargest(3, "precio")

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06


En este caso, Pandas devuelve directamente las tres filas con mayor precio.

Para los valores más bajos ocurre algo similar. Podemos ordenar de menor a mayor y usar `head()`:

In [ ]:
df.sort_values("precio").head(3)

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07
12,Goma,Libreria,300,30,Sur,Efectivo,2024-03-07
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01


O podemos usar `nsmallest()`:

In [ ]:
df.nsmallest(3, "precio")

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07
12,Goma,Libreria,300,30,Sur,Efectivo,2024-03-07
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01


Ambas estrategias pueden llegar al mismo resultado, pero no siempre se usan con la misma intención.

`sort_values()` es más general. Sirve para ordenar todo el `DataFrame`, ordenar por columnas de texto, ordenar por varias columnas o preparar una tabla completa en cierto orden.

En cambio, `nlargest()` y `nsmallest()` son más directos cuando queremos obtener los mayores o menores valores de una columna numérica.

Podemos pensarlo así:

| Queremos... | Herramienta posible |
|---|---|
| Ordenar toda la tabla | `sort_values()` |
| Ver los primeros registros de una tabla | `head()` |
| Ver los últimos registros de una tabla | `tail()` |
| Obtener los mayores valores de una columna numérica | `nlargest()` |
| Obtener los menores valores de una columna numérica | `nsmallest()` |

La elección depende de la pregunta que estemos intentando responder. Si queremos explorar el orden completo de los datos, `sort_values()` es una buena opción. Si solo queremos obtener los primeros puestos de un ranking numérico, `nlargest()` o `nsmallest()` pueden ser más directos.

## Rankings según distintas columnas

Un ranking es una lista ordenada según algún criterio.

En análisis de datos, no existe un único ranking posible. Todo depende de la variable que usemos para ordenar o seleccionar valores extremos.

Por ejemplo, si usamos la columna `precio`, podemos construir un ranking de productos más caros:

In [ ]:
df.nlargest(5, "precio")[["producto", "categoria", "precio"]]

,producto,categoria,precio
5,Calculadora,Tecnologia,18500
8,Guardapolvo,Indumentaria,14500
10,Pendrive,Tecnologia,9800
2,Mochila,Accesorios,8500
6,Auriculares,Tecnologia,7600


Esta tabla muestra las cinco ventas con mayor precio.

Pero si usamos la columna `cantidad_vendida`, el ranking cambia:

In [ ]:
df.nlargest(5, "cantidad_vendida")[["producto", "categoria", "cantidad_vendida"]]

,producto,categoria,cantidad_vendida
13,Lapiz,Libreria,50
1,Lapicera,Libreria,40
12,Goma,Libreria,30
3,Regla,Libreria,25
11,Carpeta,Libreria,18


Ahora la tabla muestra las cinco ventas con mayor cantidad de unidades vendidas.

Esto nos permite observar una diferencia importante: un producto puede estar entre los más caros, pero no necesariamente entre los más vendidos. Del mismo modo, un producto barato puede vender muchas unidades.

También podemos construir rankings de valores bajos. Por ejemplo, los cinco productos con menor precio:

In [ ]:
df.nsmallest(5, "precio")[["producto", "categoria", "precio"]]

,producto,categoria,precio
13,Lapiz,Libreria,250
12,Goma,Libreria,300
1,Lapicera,Libreria,350
3,Regla,Libreria,500
11,Carpeta,Libreria,950


O las cinco ventas con menor cantidad vendida:

In [ ]:
df.nsmallest(5, "cantidad_vendida")[["producto", "categoria", "cantidad_vendida"]]

,producto,categoria,cantidad_vendida
5,Calculadora,Tecnologia,3
8,Guardapolvo,Indumentaria,4
2,Mochila,Accesorios,5
6,Auriculares,Tecnologia,6
10,Pendrive,Tecnologia,7


Cada ranking responde una pregunta distinta.

Ordenar por `precio` nos ayuda a observar productos más caros o más baratos. Ordenar por `cantidad_vendida` nos ayuda a observar productos con mayor o menor movimiento en unidades. Más adelante, cuando aprendamos a crear nuevas columnas, podremos construir rankings más ricos, por ejemplo calculando el importe total de cada venta.

Por ahora, lo importante es reconocer que el criterio elegido cambia completamente la lectura del resultado.

## Combinar filtros y rankings

Muchas veces no queremos construir un ranking sobre todo el dataset, sino sobre un subconjunto de filas.

Por ejemplo, podemos querer conocer los productos más caros dentro de una categoría determinada. Para eso, primero filtramos las filas y luego buscamos los valores más altos.

Si queremos obtener las ventas más caras de la categoría `"Tecnologia"`, podemos escribir:

In [ ]:
df[df["categoria"] == "Tecnologia"].nlargest(3, "precio")

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04


La primera parte de la instrucción filtra el dataset:

```python
df[df["categoria"] == "Tecnologia"]
```

Después, sobre ese resultado filtrado, aplicamos:

```python
.nlargest(3, "precio")
```

De esta manera obtenemos las tres ventas con mayor precio dentro de la categoría `"Tecnologia"`.

También podemos buscar las ventas con mayor cantidad vendida en una sucursal determinada:


In [ ]:
df[df["sucursal"] == "Centro"].nlargest(5, "cantidad_vendida")

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
7,Resaltadores,Libreria,2100,12,Centro,Debito,2024-03-04
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06


En este caso, primero seleccionamos las ventas de la sucursal `"Centro"` y luego obtenemos las cinco filas con mayor cantidad vendida dentro de ese subconjunto.

También podemos combinar filtros más complejos. Por ejemplo, podemos seleccionar las ventas pagadas con `"Credito"` y buscar las de menor cantidad vendida:

In [ ]:
df[df["forma_pago"] == "Credito"].nsmallest(3, "cantidad_vendida")

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02


Esta forma de trabajo permite construir preguntas más precisas. No solo preguntamos cuáles son los valores más altos o más bajos del dataset completo, sino cuáles son los extremos dentro de un grupo específico.

La estructura general es:

```python
df[condición].nlargest(n, "columna")
```

o:

```python
df[condición].nsmallest(n, "columna")
```

Primero reducimos el dataset con un filtro. Luego buscamos los mayores o menores valores dentro de ese resultado.

## Mostrar rankings más claros

Cuando usamos `nlargest()` o `nsmallest()`, Pandas devuelve todas las columnas del `DataFrame`.

Eso puede ser útil si queremos revisar el registro completo, pero muchas veces un ranking se lee mejor si mostramos solamente las columnas necesarias.

Por ejemplo, si queremos ver los productos más caros, probablemente nos interese mostrar el producto, la categoría y el precio.

In [ ]:
df.nlargest(5, "precio")[["producto", "categoria", "precio"]]

,producto,categoria,precio
5,Calculadora,Tecnologia,18500
8,Guardapolvo,Indumentaria,14500
10,Pendrive,Tecnologia,9800
2,Mochila,Accesorios,8500
6,Auriculares,Tecnologia,7600


Esta tabla es más directa que mostrar todas las columnas. Permite concentrarse en la pregunta que estamos respondiendo: cuáles son los productos con mayor precio.

También podemos construir un ranking de productos con mayor cantidad vendida y mostrar solo las columnas relevantes:

In [ ]:
df.nlargest(5, "cantidad_vendida")[["producto", "categoria", "cantidad_vendida"]]

,producto,categoria,cantidad_vendida
13,Lapiz,Libreria,50
1,Lapicera,Libreria,40
12,Goma,Libreria,30
3,Regla,Libreria,25
11,Carpeta,Libreria,18


En este caso, el resultado se enfoca en el movimiento de unidades vendidas.

Si combinamos filtros y rankings, la selección de columnas también ayuda a que la salida sea más clara:

In [ ]:
df[df["sucursal"] == "Centro"].nlargest(
    5,
    "cantidad_vendida"
)[["producto", "sucursal", "cantidad_vendida"]]

,producto,sucursal,cantidad_vendida
13,Lapiz,Centro,50
1,Lapicera,Centro,40
0,Cuaderno,Centro,15
7,Resaltadores,Centro,12
10,Pendrive,Centro,7


Esta instrucción selecciona las ventas de la sucursal `"Centro"`, obtiene las cinco con mayor cantidad vendida y muestra solamente el producto, la sucursal y la cantidad.

Cuando construimos rankings, conviene pensar no solo en el cálculo, sino también en la presentación del resultado. Una tabla más pequeña y enfocada suele ser más fácil de interpretar que una tabla con muchas columnas.

## Errores frecuentes al buscar valores extremos

Cuando buscamos valores extremos o construimos rankings, pueden aparecer algunas confusiones.

Una primera confusión frecuente es pensar que `head()` siempre muestra los valores más altos. En realidad, `head()` solo muestra las primeras filas del `DataFrame` según el orden actual. Si el `DataFrame` no está ordenado, `head()` no necesariamente muestra un ranking.

In [ ]:
df.head()

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03


Para que `head()` muestre los valores más altos de una columna, primero debemos ordenar el `DataFrame` en forma descendente.

In [ ]:
df.sort_values("precio", ascending=False).head()

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04


Otra confusión posible es usar `tail()` sin revisar previamente el sentido del ordenamiento. Si ordenamos de menor a mayor y luego usamos `tail()`, veremos los valores más altos. Pero si ordenamos de mayor a menor y luego usamos `tail()`, veremos los valores más bajos.

In [ ]:
df.sort_values("precio").tail()

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04
2,Mochila,Accesorios,8500,5,Norte,Credito,2024-03-02
10,Pendrive,Tecnologia,9800,7,Centro,Debito,2024-03-06
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03


In [ ]:
df.sort_values("precio", ascending=False).tail()

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
11,Carpeta,Libreria,950,18,Norte,Efectivo,2024-03-06
3,Regla,Libreria,500,25,Sur,Efectivo,2024-03-02
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01
12,Goma,Libreria,300,30,Sur,Efectivo,2024-03-07
13,Lapiz,Libreria,250,50,Centro,Debito,2024-03-07


También debemos recordar que `nlargest()` y `nsmallest()` trabajan con columnas numéricas. Si queremos ordenar una columna de texto, como `producto` o `categoria`, debemos usar `sort_values()`.

Por ejemplo, esta instrucción permite ordenar alfabéticamente por producto:

In [ ]:
df.sort_values("producto")

,producto,categoria,precio,cantidad_vendida,sucursal,forma_pago,fecha
14,Agenda,Libreria,4300,9,Sur,Credito,2024-03-08
6,Auriculares,Tecnologia,7600,6,Sur,Credito,2024-03-04
5,Calculadora,Tecnologia,18500,3,Centro,Credito,2024-03-03
11,Carpeta,Libreria,950,18,Norte,Efectivo,2024-03-06
4,Cartuchera,Accesorios,3200,8,Norte,Debito,2024-03-03
9,Compas,Libreria,2600,10,Sur,Efectivo,2024-03-05
0,Cuaderno,Libreria,1200,15,Centro,Efectivo,2024-03-01
12,Goma,Libreria,300,30,Sur,Efectivo,2024-03-07
8,Guardapolvo,Indumentaria,14500,4,Norte,Credito,2024-03-05
1,Lapicera,Libreria,350,40,Centro,Debito,2024-03-01


En cambio, no tendría sentido buscar “los productos más grandes” usando `nlargest()` sobre una columna de texto.

Otro punto importante es no confundir precio alto con mayor venta total. En nuestro dataset, `precio` indica el precio unitario y `cantidad_vendida` indica cuántas unidades se vendieron. Un producto puede tener precio alto y pocas unidades vendidas, o precio bajo y muchas unidades vendidas.

Por eso, si queremos analizar ingresos, todavía nos falta una columna que calcule el importe total de cada venta. Más adelante vamos a crear nuevas columnas para responder ese tipo de preguntas.

La idea principal es que un ranking siempre depende del criterio elegido. Antes de interpretar los primeros puestos, debemos preguntarnos qué variable usamos para construir ese ranking.

## Resumen del capítulo

En este capítulo trabajamos con valores extremos y primeros rankings dentro de un `DataFrame`.

Primero retomamos `head()` y `tail()`. Vimos que `head()` muestra los primeros registros y `tail()` muestra los últimos registros según el orden actual de la tabla. Por sí solos, estos métodos no buscan valores máximos ni mínimos; simplemente muestran el comienzo o el final del `DataFrame`.

Después combinamos `sort_values()` con `head()` y `tail()`. Esta combinación nos permitió construir rankings simples. Por ejemplo, al ordenar por `precio` de mayor a menor y luego usar `head()`, pudimos obtener las ventas con precios más altos:

```python
df.sort_values("precio", ascending=False).head()
```

También vimos que el sentido del ordenamiento es fundamental. Si ordenamos de menor a mayor, los valores más bajos aparecen al principio y los más altos al final. Si ordenamos de mayor a menor, ocurre lo contrario.

Luego incorporamos dos métodos específicos para buscar valores extremos en columnas numéricas: `nlargest()` y `nsmallest()`.

Con `nlargest()` podemos obtener directamente los mayores valores de una columna:

```python
df.nlargest(3, "precio")
```

Con `nsmallest()` podemos obtener directamente los menores valores:

```python
df.nsmallest(3, "precio")
```

Estos métodos son útiles cuando queremos construir rankings numéricos de manera directa, sin ordenar visualmente todo el `DataFrame`.

También comparamos distintas estrategias. `sort_values()` es más general y sirve para ordenar tablas completas, columnas de texto o varias columnas al mismo tiempo. En cambio, `nlargest()` y `nsmallest()` son más directos cuando buscamos los mayores o menores valores de una columna numérica.

Después construimos rankings según distintas variables. Vimos que un ranking por `precio` no responde la misma pregunta que un ranking por `cantidad_vendida`. Un producto puede estar entre los más caros, pero no necesariamente entre los más vendidos. Por eso, siempre debemos interpretar un ranking según la variable usada como criterio.

Más adelante combinamos filtros y rankings. Esto nos permitió buscar valores extremos dentro de un subconjunto del dataset, por ejemplo las ventas más caras de una categoría o las ventas con mayor cantidad vendida dentro de una sucursal.

Finalmente, vimos que conviene seleccionar solamente algunas columnas para que los rankings sean más claros:

```python
df.nlargest(5, "precio")[["producto", "categoria", "precio"]]
```

La idea principal de este capítulo fue:

```text
Los rankings dependen del criterio elegido para ordenar o seleccionar valores extremos.
```

Buscar valores extremos nos ayuda a identificar casos destacados, pero la interpretación siempre debe hacerse con cuidado. Antes de sacar conclusiones, necesitamos preguntarnos qué variable usamos, qué representa esa variable y qué información todavía falta para responder mejor la pregunta de análisis.

## Próximo paso

Ya sabemos seleccionar, filtrar, ordenar y construir primeros rankings.

El siguiente paso será aprender a crear nuevas columnas dentro de un `DataFrame`. Esto nos permitirá generar información nueva a partir de datos existentes.

Por ejemplo, en nuestro dataset tenemos el precio unitario y la cantidad vendida. A partir de esas dos columnas podremos crear una nueva columna llamada `importe_total`, que represente el valor total de cada venta.

Crear columnas nuevas será un paso muy importante, porque muchas veces los datos originales no contienen directamente la variable que necesitamos analizar. En esos casos, debemos construirla.